In [3]:
import sqlite3
import numpy as np
import pandas as pd
import joblib
from scipy import stats


In [22]:
conn= sqlite3.connect("../data/inventory.db")
freight_model = joblib.load("../models/freight_model.pkl")

vendor_df= pd.read_sql_query("select * from vendor_invoice where Freight is not NULL and Quantity > 0;", conn)

X_reg= vendor_df[['Quantity','Dollars']]
vendor_df['predicted_freight'] = freight_model.predict(X_reg)
vendor_df['freight_diff'] = vendor_df['Freight'] - vendor_df['predicted_freight']
vendor_df['freight_per_unit'] = vendor_df['Freight']/vendor_df['Quantity']

risk_threshold = vendor_df['freight_diff'].quantile(0.90)
vendor_df['is_high_risk'] = (vendor_df['freight_diff']>=risk_threshold).astype(int)



In [10]:
print(f"High-Risk Overcharge Threshold: ${risk_threshold:.2f}")
print(f"Class Distribution:\n{vendor_df['is_high_risk'].value_counts(normalize=True)}")

High-Risk Overcharge Threshold: $11.92
Class Distribution:
is_high_risk
0    0.899874
1    0.100126
Name: proportion, dtype: float64


In [11]:
#Hypothesis Testing (t-test)
normal_group = vendor_df[vendor_df['is_high_risk'] == 0]
risk_group = vendor_df[vendor_df['is_high_risk'] == 1]


In [15]:
print("Statistical Significance (t-test p-values)")

for col in ['Quantity','freight_per_unit','Dollars']:
    stat, p_val = stats.ttest_ind(normal_group[col], risk_group[col], equal_var=False)
    print(f"Feature'{col}': t-stat = {stat:.2f}, p-value = {p_val:.4e}{'(Statistically Signigicant)' if p_val<0.05 else ''}")


Statistical Significance (t-test p-values)
Feature'Quantity': t-stat = -18.40, p-value = 4.9679e-60(Statistically Signigicant)
Feature'freight_per_unit': t-stat = -2.72, p-value = 6.8061e-03(Statistically Signigicant)
Feature'Dollars': t-stat = -17.03, p-value = 4.1849e-53(Statistically Signigicant)


In [23]:

import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, roc_auc_score

# 1. Prepare Features & Target
feature_cols = ['Quantity', 'Dollars', 'freight_per_unit', 'freight_diff']
X = vendor_df[feature_cols]
y = vendor_df['is_high_risk']

# 2. Train-Test Split (80% Train, 20% Test, Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Save Scaler for Inference
joblib.dump(scaler, "../models/scaler.pkl")

# 5. Evaluate Baseline Classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

print("=== Baseline Classification Models ===")
for name, clf in classifiers.items():
    X_tr = X_train_scaled if name == "Logistic Regression" else X_train
    X_te = X_test_scaled if name == "Logistic Regression" else X_test
    
    clf.fit(X_tr, y_train)
    y_pred = clf.predict(X_te)
    
    print(f"\n--- {name} ---")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
    print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")

# 6. Hyperparameter Tuning for Random Forest
param_grid = {
    'n_estimators':[50,100,150,200],
    'max_depth': [10, 20, None],
    'min_samples_split':[2,5,10]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)

print("\n=== Tuned Random Forest Results ===")
print("Best Hyperparameters:", rf_grid.best_params_)
print(f"Tuned F1-Score: {f1_score(y_test, y_pred_rf):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

# 7. Save Best Classification Model
classifier_path = "../models/risk_classifier.pkl"
joblib.dump(best_rf, classifier_path)
print(f"\nSuccessfully saved tuned Risk Classifier model to: {classifier_path}")


=== Baseline Classification Models ===

--- Logistic Regression ---
Precision: 0.9899
Recall:    0.8829
F1-Score:  0.9333

--- Decision Tree ---
Precision: 0.9911
Recall:    1.0000
F1-Score:  0.9955

--- Random Forest ---
Precision: 0.9823
Recall:    1.0000
F1-Score:  0.9911

=== Tuned Random Forest Results ===
Best Hyperparameters: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 50}
Tuned F1-Score: 0.9911

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       998
           1       0.98      1.00      0.99       111

    accuracy                           1.00      1109
   macro avg       0.99      1.00      1.00      1109
weighted avg       1.00      1.00      1.00      1109


Successfully saved tuned Risk Classifier model to: ../models/risk_classifier.pkl


In [24]:
pip install streamlit

  Using cached pillow-12.3.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 349.5 kB/s eta 0:00:00 0:00:01
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached idna-3.19-py3-none-any.whl.metadata (9.2 kB)
  Using cached charset_normalizer-3.5.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (45 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
 